# Preprocessing

In [1]:
import re 
import time
import os
import random
from concurrent.futures import ThreadPoolExecutor


import pandas as pd
import xarray as xr
from xarray.backends.api import open_datatree
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pyproj
import boto3
import earthaccess

from dotenv import load_dotenv

load_dotenv()

auth = earthaccess.login()

In [2]:
tspan = ('2025-09-02T00:00:00Z', '2025-09-04T00:00:00Z')

results = earthaccess.search_data(
    short_name = "PACE_OCI_L1B_SCI",
    temporal=tspan,
    count=10000
)

print("total results: ", len(results))

total results:  289


# Iteration 1: Save Entire Granule

In [3]:
granules = ['20250903T144236', '20250903T144736','20250903T145236','20250903T145736', '20250903T150236']

scan_ids = range(1065)

def store_granule(granule_id, scan_ids=scan_ids):
  print("++++++++++++")
  print(f"Processing granule {granule_id}")
  results = earthaccess.search_data(
    short_name="PACE_OCI_L1B_SCI",
    granule_name=f"PACE_OCI.{granule_id}.L1B.V3.nc",)

  paths = earthaccess.open(results)
  datatree = open_datatree(paths[0])
  dataset = xr.merge(datatree.to_dict().values())

  for scan_id in scan_ids:
    get_tostore_vector(dataset, scan_id, granule_id)
    if not scan_id % 50:
      print(f"scan_id: {scan_id} completed for granule {granule_id}")


def get_tostore_vector(dataset, scan_id, granule_id):
  rhot_blue_arr = dataset["rhot_blue"].isel(scans=scan_id).values
  rhot_red_arr = dataset["rhot_red"].isel(scans=scan_id).values
  rhot_SWIR_arr = dataset["rhot_SWIR"].isel(scans=scan_id).values

  tostore = np.vstack([rhot_blue_arr, rhot_red_arr, rhot_SWIR_arr]).T

  path = f"s3://compressive-sensing/{granule_id}/scan{scan_id}.parquet"

  df = pd.DataFrame(tostore)
  df.to_parquet(
      path,
      index=False,
      engine="pyarrow",
  )


# 2. Sampling - 1% of Granules

In [ ]:

granules = [re.search(r"(\d{8}T\d{6})", res.dataviz_links()[0]).group(1) for res in results]

data = []

def process_granule(granule_id):
    results = earthaccess.search_data(
    short_name="PACE_OCI_L1B_SCI",
    granule_name=f"PACE_OCI.{granule_id}.L1B.V3.nc",)

    random_scan = random.randint(0, 1065)

    print(f"Processing granule {granule_id} with random scan {random_scan}")

    paths = earthaccess.open(results)
    datatree = open_datatree(paths[0])
    dataset = xr.merge(datatree.to_dict().values())

    rhot_blue_arr = dataset["rhot_blue"].isel(scans=random_scan).values
    rhot_red_arr = dataset["rhot_red"].isel(scans=random_scan).values
    rhot_SWIR_arr = dataset["rhot_SWIR"].isel(scans=random_scan).values
    tostore = np.vstack([rhot_blue_arr, rhot_red_arr, rhot_SWIR_arr]).T

    path = f"s3://compressive-sensing/samples/{granule_id}/scan{random_scan}.parquet"

    df = pd.DataFrame(tostore)
    df.to_parquet(
        path,
        index=False,
        engine="pyarrow",
    )

    return path

with ThreadPoolExecutor(max_workers=10) as executor:
  results = executor.map(process_granule, granules[:5])
  for result in results:
    print("Stored object at: ",result)

data = np.array(data)
print(data.shape)

    

Processing granule 20250902T004050 with random scan 1062


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

Processing granule 20250902T004550 with random scan 335


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]